In [0]:
%run /Workspace/Users/valterlafuentejunior@gmail.com/IngestaodedadosAPInoDatabricks/Config/config

In [0]:
import requests
import yfinance as yf
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession

In [0]:
def buscar_dados_acao(ticker_b3):
    """
    Coleta dados diários de uma ação da B3 via Yahoo Finance e
    retorna um DataFrame com schema fixo e colunas genéricas.
    """
    try:
        ticker = ticker_b3 + ".SA"
        print(f"📊 Coletando dados do ticker: {ticker}...")

        # Faz o download
        df = yf.download(ticker, period="6mo", interval="1d", group_by="ticker")

        if df.empty:
            print(f"⚠️ Nenhum dado encontrado para {ticker_b3}.")
            return None

        # Se o DataFrame vier com MultiIndex (ex: ('PETR4.SA', 'Open')), achatamos
        if isinstance(df.columns, pd.MultiIndex):
            # Extrai apenas o segundo nível (Open, High, Low, Close, Volume)
            df.columns = df.columns.get_level_values(1)

        # Agora renomeia com segurança (se existirem)
        df.reset_index(inplace=True)

        colunas_atuais = [c.lower() for c in df.columns]
        rename_map = {}
        if "open" in colunas_atuais: rename_map[df.columns[colunas_atuais.index("open")]] = "Abertura"
        if "high" in colunas_atuais: rename_map[df.columns[colunas_atuais.index("high")]] = "Alta"
        if "low" in colunas_atuais: rename_map[df.columns[colunas_atuais.index("low")]] = "Baixa"
        if "close" in colunas_atuais: rename_map[df.columns[colunas_atuais.index("close")]] = "Fechamento"
        if "volume" in colunas_atuais: rename_map[df.columns[colunas_atuais.index("volume")]] = "Volume"
        if "date" in colunas_atuais: rename_map[df.columns[colunas_atuais.index("date")]] = "data"

        df.rename(columns=rename_map, inplace=True)

        # Garante que apenas as colunas desejadas existam
        colunas_finais = ["data", "Abertura", "Alta", "Baixa", "Fechamento", "Volume"]
        df = df[[c for c in colunas_finais if c in df.columns]]

        # Adiciona metadados
        df["ticker"] = ticker_b3
        df["data_ingestao"] = datetime.now()

        print(f"✅ {ticker_b3} coletado com sucesso!")
        return df

    except Exception as e:
        print(f"❌ Erro ao coletar {ticker_b3}: {e}")
        return None

In [0]:
dfs = []
for ticker in tickets:
    df = buscar_dados_acao(ticker)
    if df is not None:
        dfs.append(df)
    else:
        print(f"⚠️ Falha ao coletar dados do ticker {ticker}")

# Garante que existam dados
if not dfs:
    raise ValueError("❌ Nenhum dado retornado pelo Yahoo Finance.")

# Concatena todos os DataFrames (um único schema)
df_final = pd.concat(dfs, ignore_index=True)

# ======================================================
# 🔹 Converte para Spark e grava no Delta Lake
# ======================================================
df_spark = spark.createDataFrame(df_final)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

tabela_existe = spark.catalog.tableExists("workspace.bronze.cotacoes_yfinance")

if not tabela_existe:
    print("🆕 Criando tabela 'workspace.bronze.cotacoes_yfinance'...")
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.bronze.cotacoes_yfinance")
    )
    print("✅ Tabela criada com sucesso!")
else:
    print("📥 Inserindo novos dados...")
    (
        df_spark.write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.bronze.cotacoes_yfinance")
    )
    print("✅ Dados adicionados com sucesso!")


In [0]:
df_spark.distinct().toPandas()